# 2. Two logical qubits and what braiding can do

The 4 ground states of the torus are a register of two qubits, $|q_1 q_2\rangle$. This notebook shows how to read
them, how to apply gates by moving anyons, and why the toric code alone cannot run arbitrary quantum computations.

In [1]:
import numpy as np

import toric

ex = toric.ExactToricCode(3)
t = ex.torus

## Reading the qubits: loops around the torus

$q_1$ is read by a $Z$ loop along $x$ and $q_2$ by a $Z$ loop along $y$: they give $(-1)^{q_1}$ and $(-1)^{q_2}$.
Moving such a loop by one row multiplies it by the plaquettes in between, all $+1$ in a ground state, so it does not
matter which row you use. No smaller operator can read the qubits: that is why they are protected.

In [2]:
for z, label in enumerate(["|00>", "|01>", "|10>", "|11>"]):
    psi = ex.basis[z]
    rows = [ex.expectation("Z", t.z_row(r), psi) for r in range(3)]
    cols = [ex.expectation("Z", t.z_col(c), psi) for c in range(3)]
    print(f"{label}: Z loop along x on rows 0,1,2 = {np.round(rows, 6)},  along y on columns = {np.round(cols, 6)}")

|00>: Z loop along x on rows 0,1,2 = [1. 1. 1.],  along y on columns = [1. 1. 1.]
|01>: Z loop along x on rows 0,1,2 = [1. 1. 1.],  along y on columns = [-1. -1. -1.]
|10>: Z loop along x on rows 0,1,2 = [-1. -1. -1.],  along y on columns = [1. 1. 1.]
|11>: Z loop along x on rows 0,1,2 = [-1. -1. -1.],  along y on columns = [-1. -1. -1.]


## Gates are anyons dragged around the torus

| Gate | Move | Effect |
|---|---|---|
| $\bar X_1$ | an m around the torus along $y$ ($X$ across a column) | flips $q_1$ |
| $\bar X_2$ | an m around the torus along $x$ | flips $q_2$ |
| $\bar Z_1$ | an e around the torus along $x$ ($Z$ along a row) | $-1$ on the $q_1 = 1$ part |
| $\bar Z_2$ | an e around the torus along $y$ | $-1$ on the $q_2 = 1$ part |

Why is $\bar Z_1$ a *braid*? In the $|q_1 = 1\rangle$ part of the state an m has gone around $y$, so one unit of m
flux threads the loop along $x$. Taking an e around $x$ encircles that flux: $-1$, the e–m braiding phase of
notebook 1. In the $|q_1 = 0\rangle$ part there is no flux and no phase.

In [3]:
plus0 = np.array([1, 0, 1, 0]) / np.sqrt(2)          # |+0>
s = toric.new_state(6)
s.apply_string("Z", s.torus.z_row(3))                 # an e around x: Zbar_1
print("|+0>  ->", np.round(s.logical(plus0), 3), " = |-0>")
s.apply_string("X", s.torus.x_row(2))                 # an m around x: Xbar_2
print("then Xbar_2 ->", np.round(s.logical(plus0), 3), " = |-1>")

|+0>  -> [ 0.707+0.j  0.   +0.j -0.707+0.j -0.   +0.j]  = |-0>
then Xbar_2 -> [ 0.   +0.j  0.707+0.j -0.   +0.j -0.707+0.j]  = |-1>


## What braiding cannot do

Toric-code anyons are **abelian**: braiding only multiplies by $\pm 1$. The gates above are Pauli operators and
nothing more. A Pauli never turns a basis state into a superposition, and never creates entanglement: check it on
all 16 products of the four gates.

In [4]:
def concurrence(c):
    return 2 * abs(c[0] * c[3] - c[1] * c[2])

paulis = [(a, b, u, w) for a in (0, 1) for b in (0, 1) for u in (0, 1) for w in (0, 1)]
for name, c in [("|00>", [1, 0, 0, 0]), ("|+0>", [1, 0, 1, 0] / np.sqrt(2)), ("Bell", [1, 0, 0, 1] / np.sqrt(2))]:
    C = []
    for a, b, u, w in paulis:
        s = toric.new_state(4)
        T = s.torus
        if a:
            s.apply_string("X", T.x_col(1))
        if b:
            s.apply_string("X", T.x_row(1))
        if u:
            s.apply_string("Z", T.z_row(1))
        if w:
            s.apply_string("Z", T.z_col(1))
        C.append(concurrence(s.logical(c)))
    print(f"start {name}: concurrence after each of the 16 gates = {sorted({float(round(x, 6)) for x in C})}")

start |00>: concurrence after each of the 16 gates = [0.0]
start |+0>: concurrence after each of the 16 gates = [0.0]
start Bell: concurrence after each of the 16 gates = [1.0]


Entanglement is never created and never destroyed; from a Bell state the gates only move you among the four Bell
states. Computing by braiding alone needs **non-abelian** anyons (for example Fibonacci anyons), where exchanging two
anyons applies a matrix on a degenerate space rather than a phase.

## Exercises

1. Prepare all four basis states from $|00\rangle$ using only $X$ loops. Check them with the $Z$ loops.
2. Start from the Bell state $(|00\rangle + |11\rangle)/\sqrt 2$. Which gate turns it into $(|01\rangle + |10\rangle)/\sqrt 2$?
3. Apply $\bar Z_1$ to $|00\rangle$. Did anything physical change? What about $|10\rangle$?